In [1]:
import torch

pt = torch.load('/home/scpark/data/samplings/DiT/1.5/200/Euler/1000/train_traj_4/0.pt')
pt.keys()

dict_keys(['cond', 'sample', 'noise', 'traj', 'timesteps'])

In [2]:
pt['traj'].shape

torch.Size([201, 4, 32, 32])

In [7]:
import torch

# 당신이 만든 함수
def interp_traj(X, t, s):
    # X:(B,L,C,H,W), t,s: (L,),(M,) — 둘 다 내림차순(1→0) 가정
    t, s = t.to(X.device), s.to(X.device)
    i1 = torch.bucketize(-s, -t).clamp(1, t.numel()-1); i0 = i1 - 1
    w  = ((s - t[i0]) / (t[i1] - t[i0])).to(X.dtype).view(1, -1, 1, 1, 1)
    return torch.lerp(X[:, i0], X[:, i1], w)


In [8]:
teacher_traj = pt['traj'].unsqueeze(0)
print(teacher_traj.shape)
teacher_timesteps = pt['timesteps']
print(teacher_timesteps.shape)
student_timesteps = torch.tensor([0.998, 0.77, 0.44, 0.33, 0.0])
student_traj = interp_traj(teacher_traj, teacher_timesteps, student_timesteps)
print(student_traj.shape)

torch.Size([1, 201, 4, 32, 32])
torch.Size([201])
torch.Size([1, 5, 4, 32, 32])


In [9]:

# ========= 검증 =========
def check_interp_traj():
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    B, L, C, H, W = 2, 7, 3, 4, 5
    t = torch.linspace(1, 0, L, device=dev)            # 내림차순
    X = torch.randn(B, L, C, H, W, device=dev)

    # 1) identity: s == t -> 그대로 복원
    Y = interp_traj(X, t, t)
    assert torch.allclose(Y, X, atol=0, rtol=0)

    # 2) 선형성: X를 시간 t 자체로 채우면, 보간 결과는 s가 되어야 함
    X_lin = t.view(1, L, 1, 1, 1).expand(B, L, C, H, W)
    s = torch.tensor([1.0, 0.75, 0.5, 0.25, 0.0], device=dev)  # 내림차순
    Y = interp_traj(X_lin, t, s)
    target = s.view(1, -1, 1, 1, 1).expand_as(Y)
    assert torch.allclose(Y, target, atol=1e-6)

    # 3) 모양/범위: 임의 s (내림차순, 범위 내부)
    M = 11
    s = torch.linspace(1, 0, M, device=dev)
    Y = interp_traj(X, t, s)
    assert Y.shape == (B, M, C, H, W)

    # 4) 역전파 스모크: 그래디언트가 잘 흐르는지
    X_req = torch.randn(B, L, C, H, W, device=dev, requires_grad=True)
    Y = interp_traj(X_req, t, s)
    Y.sum().backward()
    assert torch.isfinite(X_req.grad).all()

    print("✅ interp_traj: all sanity checks passed")

check_interp_traj()


✅ interp_traj: all sanity checks passed
